#### Imports

In [2]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

# Import SciPy and NumPy
from scipy.stats import t  # We only need the t class from scipy.stats
import numpy as np

import statsmodels.api as sm
from statsmodels.stats.weightstats import ztest

import pandas as pd
from scipy.stats import spearmanr
from scipy.stats import kendalltau

In [3]:
mob_mtx_filename = 'data/mobilidade.csv'
dist_mtx_filename = 'data/distances.xlsx'
infmutua_filename = 'data/weeklycases_mutualinformation.csv'

In [4]:
distance_mtx = dist_mtx_filename
mob_mtx = mob_mtx_filename

In [5]:
#Eliminating unnecessary rows and columns from matrix
dist = pd.read_excel(distance_mtx, index_col = 'ZT')
dist.columns = dist.columns.astype(int)
indexes_to_drop = [36, 41, 50, 52, 53, 55] #Vazios urbanos ['36', '41', '50', '52', '55', '53']
dist = dist.drop(columns=indexes_to_drop, index = indexes_to_drop)


indexes_to_drop2 = [35, 54]
dist = dist.drop(columns=indexes_to_drop2, index = indexes_to_drop2)# dist = dist.drop(columns=53, index=53)
dist.columns = dist.index
dist

ZT,1,2,3,4,5,6,7,8,9,10,...,40,42,43,44,45,46,47,48,49,51
ZT,,,,,,,,,,,,,,,,,,,,,
1,0.000000,1476.718825,1576.908816,1789.142336,870.921123,1504.660907,1875.497068,3037.629836,3222.811990,3200.724483,...,5374.083056,10961.879275,9906.424483,10674.120724,12401.840419,12400.597015,10437.861320,21287.463724,3824.743821,9685.437253
2,1476.718825,0.000000,1012.275199,1777.127182,1986.767509,2980.023255,3163.703621,2159.615749,2577.565242,3639.298199,...,4077.545081,9978.710582,9188.215627,10481.715732,11861.307317,12987.762392,11884.044322,20281.885280,3794.083713,10035.409487
3,1576.908816,1012.275199,0.000000,800.646443,1585.073554,2938.325517,3452.389330,3165.137212,1722.895254,2673.784378,...,3862.671573,9397.788339,8426.396500,9520.637775,11016.910560,12001.660513,11495.576491,21184.955385,2785.010803,9025.786284
4,1789.142336,1777.127182,800.646443,0.000000,1368.407672,2832.145711,3569.213565,3936.086821,1542.322142,1874.204549,...,4179.644855,9298.352815,8138.613592,8943.717888,10612.711087,11212.467566,10962.789061,21984.348696,2114.325379,8270.870313
5,870.921123,1986.767509,1585.073554,1368.407672,0.000000,1463.781609,2273.097117,3841.185903,2910.564826,2388.286033,...,5421.713554,10662.811264,9446.756994,9968.673525,11833.899959,11530.745942,9956.897877,22101.057680,3155.568689,8841.202364
6,1504.660907,2980.023255,2938.325517,2832.145711,1463.781609,0.000000,1087.829050,4241.077369,4374.102010,3500.589423,...,6799.389390,12124.287862,10866.296436,11156.039100,13178.439703,12050.109109,9000.398548,22302.296140,4475.061198,9643.831992
7,1875.497068,3163.703621,3452.389330,3569.213565,2273.097117,1087.829050,0.000000,3832.983317,5069.606900,4523.640575,...,7206.182917,12827.361960,11702.810949,12176.879400,14106.193060,13118.782364,9381.974686,21547.975646,5416.028359,10731.252742
8,3037.629836,2159.615749,3165.137212,3936.086821,3841.185903,4241.077369,3832.983317,0.000000,4534.533604,5789.144197,...,4973.885515,11198.481342,10775.431471,12466.572264,13584.193010,15141.618823,13181.329330,18260.026454,5923.911202,12190.499231
9,3222.811990,2577.565242,1722.895254,1542.322142,2910.564826,4374.102010,5069.606900,4534.533604,0.000000,2378.093544,...,2896.356816,7765.770856,6706.204069,7940.217834,9303.908119,11080.515900,12200.372806,21921.637835,1698.407879,7899.922834


In [6]:
infmutua = pd.read_csv(infmutua_filename, index_col=0)
infmutua

,1,2,3,4,5,6,7,8,9,10,...,40,42,43,44,45,46,47,48,49,51
1,1.000000,0.336794,0.565065,0.635834,0.210062,0.231241,0.286893,0.255507,0.183230,0.275626,...,0.094385,0.336337,0.254972,0.268289,0.325170,0.052719,0.254294,0.000000,0.168031,0.004604
2,0.441903,1.000000,0.507982,0.594110,0.377482,0.429704,0.374320,0.258945,0.158562,0.286282,...,0.072302,0.443751,0.260484,0.457028,0.206370,0.234240,0.286278,0.020410,0.148683,0.012536
3,0.526039,0.457575,1.000000,0.564924,0.304358,0.384684,0.447871,0.363447,0.308044,0.326478,...,0.144133,0.477951,0.375917,0.371733,0.244015,0.124355,0.393781,0.074383,0.204457,0.057623
4,0.721088,0.508158,0.629607,1.000000,0.368378,0.298210,0.308838,0.245121,0.178637,0.272804,...,0.041090,0.334461,0.225853,0.269330,0.215244,0.097853,0.267384,0.000000,0.147063,0.000000
5,0.229346,0.274948,0.227467,0.384370,1.000000,0.258008,0.220814,0.093478,0.092501,0.196765,...,0.094310,0.233716,0.123074,0.127523,0.247991,0.164341,0.221438,0.061964,0.092119,0.136065
6,0.201741,0.248487,0.234140,0.330686,0.250359,1.000000,0.367106,0.255235,0.370982,0.341043,...,0.156586,0.370388,0.261695,0.320029,0.181087,0.043553,0.289883,0.077694,0.222624,0.161439
7,0.285496,0.286188,0.324597,0.395190,0.233272,0.216484,1.000000,0.324605,0.194266,0.278567,...,0.091415,0.430675,0.293540,0.225177,0.182867,0.111035,0.293975,0.018565,0.109012,0.056497
8,0.315013,0.320816,0.344840,0.457283,0.134161,0.334640,0.379623,1.000000,0.252263,0.328817,...,0.096665,0.443513,0.275201,0.262461,0.207742,0.044923,0.280289,0.052218,0.126609,0.062136
9,0.291029,0.336160,0.319676,0.443440,0.311998,0.430514,0.467761,0.408423,1.000000,0.452739,...,0.147101,0.507993,0.356972,0.361964,0.227091,0.053183,0.406135,0.051692,0.207150,0.169728
10,0.319089,0.354679,0.369778,0.434024,0.331513,0.363884,0.406706,0.403423,0.466421,1.000000,...,0.139183,0.554398,0.431034,0.487954,0.300735,0.021994,0.494019,0.034601,0.311087,0.135178


In [8]:
mob = pd.read_csv(mob_mtx, index_col='ZT')
mob.columns = mob.columns.astype(int)

# indexes_to_drop2 = [35, 54]
# mob = mob.drop(columns=indexes_to_drop2, index = indexes_to_drop2)
mob

,1,2,3,4,5,6,7,8,9,10,...,40,42,43,44,45,46,47,48,49,51
ZT,,,,,,,,,,,,,,,,,,,,,
1,12551.0,8033.0,4295.0,2859.0,3332.0,9223.0,7266.0,5575.0,9510.0,3466.0,...,482.0,4560.0,4545.0,9884.0,500.0,0.0,1956.0,896.0,516.0,175.0
2,8033.0,7455.0,1194.0,301.0,925.0,2339.0,3636.0,3441.0,2095.0,388.0,...,120.0,500.0,136.0,881.0,642.0,222.0,439.0,208.0,69.0,73.0
3,4295.0,1194.0,4170.0,2562.0,2109.0,600.0,894.0,553.0,2003.0,1940.0,...,124.0,929.0,606.0,896.0,95.0,0.0,40.0,0.0,246.0,44.0
4,2859.0,301.0,2562.0,2817.0,322.0,766.0,820.0,164.0,660.0,117.0,...,16.0,410.0,0.0,54.0,0.0,0.0,0.0,0.0,18.0,0.0
5,3332.0,925.0,2109.0,322.0,1382.0,1533.0,1185.0,501.0,1320.0,913.0,...,55.0,1233.0,542.0,440.0,48.0,98.0,279.0,0.0,66.0,0.0
6,9223.0,2339.0,600.0,766.0,1533.0,5654.0,9912.0,837.0,1506.0,836.0,...,101.0,743.0,502.0,941.0,142.0,0.0,80.0,180.0,39.0,0.0
7,7266.0,3636.0,894.0,820.0,1185.0,9912.0,8334.0,1846.0,1876.0,600.0,...,24.0,203.0,203.0,1430.0,48.0,0.0,160.0,232.0,116.0,0.0
8,5575.0,3441.0,553.0,164.0,501.0,837.0,1846.0,11188.0,1517.0,117.0,...,54.0,287.0,136.0,853.0,48.0,111.0,120.0,90.0,13.0,58.0
9,9510.0,2095.0,2003.0,660.0,1320.0,1506.0,1876.0,1517.0,10952.0,434.0,...,960.0,4011.0,1743.0,3417.0,832.0,0.0,280.0,313.0,472.0,132.0


#### Distance X Mutual Information

In [9]:
def get_upper_triangular(df):
    upper_triangular_values = []

    for i in range(len(df)):
        for j in range(i + 1, len(df.columns)):
            value = df.iloc[i, j]
            upper_triangular_values.append(value)
            if value < 0:
                row_label = df.index[i]
                col_label = df.columns[j]
                print(f'Valor menor que 0 encontrado na linha {i} (ZT: {row_label}) e coluna {j} (ZT: {col_label})')
    return upper_triangular_values

upper_tri_matrix1 = get_upper_triangular(infmutua)
upper_tri_matrix2 = get_upper_triangular(dist)

distance_infmutua_df = pd.DataFrame({'dist': upper_tri_matrix2, 'mutual_inf': upper_tri_matrix1})

distance_infmutua_df

,dist,mutual_inf
0,1476.718825,0.336794
1,1576.908816,0.565065
2,1789.142336,0.635834
3,870.921123,0.210062
4,1504.660907,0.231241
...,...,...
1076,11108.738894,0.226692
1077,11275.159681,0.018817
1078,23603.176668,0.028346
1079,29752.187623,0.091242


In [10]:
dist_versus_infmutua = distance_infmutua_df.groupby(distance_infmutua_df.columns.tolist()).size().reset_index().\
    rename(columns={0:'count'})
dist_versus_infmutua.head()

,dist,mutual_inf,count
0,248.453551,0.458424,1
1,800.646443,0.564924,1
2,870.921123,0.210062,1
3,1007.553485,0.543277,1
4,1012.275199,0.507982,1


In [11]:
dist_versus_infmutua.sort_values(by='mutual_inf')

,dist,mutual_inf,count
1001,16498.289694,0.000000,1
271,5283.187630,0.000000,1
279,5372.824509,0.000000,1
1048,21984.348696,0.000000,1
1045,21287.463724,0.000000,1
...,...,...,...
935,14135.911662,0.608223,1
297,5559.025153,0.608523,1
1028,18491.083505,0.632321,1
33,1789.142336,0.635834,1


In [12]:
num_splits=40
dfs = np.array_split(dist_versus_infmutua, num_splits)
mins= []
maxs= []
means= []
medians = []
for i in range(num_splits):
  mins.append(dfs[i]['mutual_inf'].min())
  maxs.append(dfs[i]['mutual_inf'].max())
  means.append(dfs[i]['mutual_inf'].mean())
  medians.append(dfs[i]['dist'].median())

d1 = {'mins':mins,'maxs':maxs,'means':means,'medians':medians}
data1 = pd.DataFrame(d1)

data1.head()

/Users/catiasepetauskas/miniconda3/envs/test_env_310/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


,mins,maxs,means,medians
0,0.000000,0.565065,0.328887,1467.075100
1,0.000000,0.635834,0.293596,2016.023194
2,0.020258,0.523228,0.305612,2565.422813
3,0.074747,0.652925,0.304273,3105.278394
4,0.000000,0.557097,0.260736,3476.783102


In [13]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=dist_versus_infmutua['dist'], y=dist_versus_infmutua['mutual_inf'],
                    mode='markers', name='Original Data',
                     hovertemplate =
                    '<i>Distance</i>: %{x}<br>'+
                    '<i>Max Corr</i>: %{y}<br>'+
                    '<i>Count</i>: %{marker.size:,}',
                    marker=dict(
                       opacity=0.25,
                       color=dist_versus_infmutua['mutual_inf'],
                       colorbar=dict(
                          title="Mutual<br>Information"
                       ),
                       size=dist_versus_infmutua['count'],
                       sizemode='area',
                       sizeref=2.*(dist_versus_infmutua['count'].max())/(20.**2),
                       sizemin=1,
                       colorscale="Bluered"
                    )))
fig2.update_layout(legend= {'itemsizing': 'constant'})

fig2.add_trace(go.Scatter(x=data1['medians'], y=data1['maxs'],
                    mode='lines+markers', name='max', line=dict(color='rgb(130,8,8)', width=2)))
fig2.add_trace(go.Scatter(x=data1['medians'], y=data1['means'],
                    mode='lines+markers', name='mean', line=dict(color='rgb(100,50,100)', width=2)))
fig2.add_trace(go.Scatter(x=data1['medians'], y=data1['mins'],
                    mode='lines+markers', name='min', line=dict(color='rgb(8,8,130)', width=2)))


# fig2.update_xaxes(type="log")
fig2.update_layout(
    title_text="<b>Distance x Mutual Information</b>",
    xaxis_title="<b>Distance(m)</b>",
    yaxis_title="<b>Mutual Information</b>",
    font=dict(
        family="Tahoma",
        size=25,
      #   color="RebeccaPurple"
    ),
    width=1000,
    height=600,
    legend=dict(
      yanchor="top",
      y=0.99,
      xanchor="left",
      x=0.8,
      font=dict(
          family="Tahoma",
          size=16,  # Ajuste o tamanho da fonte da legenda aqui
          color="black"
      )
    ),
    hovermode="closest"
   )

fig2.show()

#### Distance X Mutual Information

In [14]:
upper_tri_matrix1 = get_upper_triangular(mob)
upper_tri_matrix2 = get_upper_triangular(infmutua)

# Criando um dataframe a partir das listas
mob_corr_df = pd.DataFrame({'mob': upper_tri_matrix1, 'mutual_inf': upper_tri_matrix2})

mob_corr_df

,mob,mutual_inf
0,8033.0,0.336794
1,4295.0,0.565065
2,2859.0,0.635834
3,3332.0,0.210062
4,9223.0,0.231241
...,...,...
1076,0.0,0.226692
1077,0.0,0.018817
1078,13.0,0.028346
1079,0.0,0.091242


#### Only Mobility Values Greater Than 0 Were Considered

In [15]:
mob_corr_df = mob_corr_df[mob_corr_df.mob != 0]
mob_corr_df

,mob,mutual_inf
0,8033.0,0.336794
1,4295.0,0.565065
2,2859.0,0.635834
3,3332.0,0.210062
4,9223.0,0.231241
...,...,...
1064,453.0,0.281334
1065,88.0,0.080029
1069,48.0,0.073674
1078,13.0,0.028346


In [16]:
import math

max_mob = mob_corr_df.loc[mob_corr_df['mob'].idxmax()]

# limit = limit_mob
ceil = math.ceil(max_mob['mob'])

max_corrs = []
min_corrs = []
mean_corrs = []

for i in range(0, ceil):
  interval = mob_corr_df[mob_corr_df['mob'].between(i, i)]

  if(len(interval)>0):
    corr_mean = interval['mutual_inf'].mean()
    mob_mean = interval['mob'].mean()
    mean_corrs.append((mob_mean, corr_mean))

    max = interval.loc[interval['mutual_inf'].idxmax()]
    min = interval.loc[interval['mutual_inf'].idxmin()]
    max_corrs.append((mob_mean, max['mutual_inf']))
    min_corrs.append((mob_mean, min['mutual_inf']))

mins_dataframe = pd.DataFrame(min_corrs)
maxs_dataframe = pd.DataFrame(max_corrs)
means_dataframe = pd.DataFrame(mean_corrs)

means_dataframe.rename(columns={0: 'mob', 1: 'mutual_inf', 2: 'num_elements'}, inplace=True)
maxs_dataframe.rename(columns={0: 'mob', 1: 'mutual_inf'}, inplace=True)
mins_dataframe.rename(columns={0: 'mob', 1: 'mutual_inf'}, inplace=True)

#### Mobility X Correlation - using plotly


In [17]:
mob_versus_infmutua = mob_corr_df.groupby(mob_corr_df.columns.tolist()).size().reset_index().\
    rename(columns={0:'count'})
mob_versus_infmutua.head()

,mob,mutual_inf,count
0,3.0,0.000000,1
1,3.0,0.057389,1
2,6.0,0.027586,1
3,6.0,0.068637,1
4,6.0,0.069676,1


In [18]:
mob_versus_infmutua.sort_values(by='mutual_inf')

,mob,mutual_inf,count
0,3.0,0.000000,1
296,210.0,0.000000,1
80,48.0,0.000000,1
453,468.0,0.000000,1
98,59.0,0.000000,1
...,...,...,...
428,424.0,0.607280,1
25,21.0,0.608523,1
318,222.0,0.632321,1
752,2859.0,0.635834,1


In [19]:
num_splits=40
dfs = np.array_split(mob_versus_infmutua, num_splits)
mins= []
maxs= []
means= []
medians = []
for i in range(num_splits):
  mins.append(dfs[i]['mutual_inf'].min())
  maxs.append(dfs[i]['mutual_inf'].max())
  means.append(dfs[i]['mutual_inf'].mean())
  medians.append(dfs[i]['mob'].median())

d2 = {'mins':mins,'maxs':maxs,'means':means,'medians':medians}
data2 = pd.DataFrame(d2)

data2.head()

/Users/catiasepetauskas/miniconda3/envs/test_env_310/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning:

'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.



,mins,maxs,means,medians
0,0.000000,0.270746,0.107843,8.0
1,0.000000,0.608523,0.136771,26.0
2,0.005038,0.394801,0.187465,38.0
3,0.000000,0.530342,0.222204,44.0
4,0.000000,0.530488,0.211796,54.0


In [20]:
import plotly.express as px

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=mob_versus_infmutua['mob'], y=mob_versus_infmutua['mutual_inf'],
                    mode='markers', name='Original Data',
                     hovertemplate =
                    '<i>Mobility</i>: %{x}<br>'+
                    '<i>Corr</i>: %{y}<br>'+
                    '<i>Count</i>: %{marker.size:,}',
                    marker=dict(
                       opacity=0.25,
                       color=mob_versus_infmutua['mutual_inf'],
                       colorbar=dict(
                          title="Mutual<br>Information"
                       ),
                       size=mob_versus_infmutua['count'],
                       sizemode='area',
                       sizeref=2.*(mob_versus_infmutua['count'].max())/(20.**2),
                       sizemin=1,
                       colorscale="Bluered"
                    )))
fig2.update_layout(legend= {'itemsizing': 'constant'})

fig2.add_trace(go.Scatter(x=data2['medians'], y=data2['maxs'],
                    mode='lines+markers', name='max', line=dict(color='rgb(130,8,8)', width=2)))
fig2.add_trace(go.Scatter(x=data2['medians'], y=data2['means'],
                    mode='lines+markers', name='mean', line=dict(color='rgb(100,50,100)', width=2)))
fig2.add_trace(go.Scatter(x=data2['medians'], y=data2['mins'],
                    mode='lines+markers', name='min', line=dict(color='rgb(8,8,130)', width=2)))

fig2.update_xaxes(type="log")
fig2.update_layout(
    title_text="<b>Mobility x Mutual Information</b>",
    xaxis_title="<b>Mobility (log)</b>",
    yaxis_title="<b>Mutual Information</b>",
    font=dict(
        family="Tahoma",
        size=25,
      #   color="RebeccaPurple"
    ),
    width=1000,
    height=600,
    legend=dict(
      yanchor="top",
      y=0.99,
      xanchor="left",
      x=0.02,
      font=dict(
          family="Tahoma",
          size=16,  # Ajuste o tamanho da fonte da legenda aqui
          color="black"
      )
    ),
    hovermode="closest"
   )

fig2.show()

# Linear Regression

Therefore, a linear regression was performed to assess the possible linearity relationship between correlation x distance" and correlation x mobility". This method consists of estimating parameters b0 and b 1 of the following linear equation (X): Y = b1x+b0.

For the statistical treatment of data on distance, mobility, and correlation of Dengue cases presented in this work, a hypothesis test was developed for the parameters calculated from the linear regression line of the data. The most conservative hypothesis, H0, states that there is no statistically signi cant linear relationship between the 2 variables studied and which is sought to be rejected. Therefore, a useful approach is to use the p-value or probability of signi cance. The p-value is a statistic commonly used to synthesize the result of a hypothesis test and indicates the probability of the test statistic being equal to or more extreme than the sample statistic, that is, of being in the rejection region of H0, assuming it is true. If p < alpha , then a type I error occurred and H0 is rejected in favor of the alternative hypothesis H1, more innovative.

In [21]:
def perform_regression_and_ztest(dataframe, variavel1, variavel2):
    # Adjust the OLS Regression Model
    X = sm.add_constant(dataframe[variavel1])
    y = dataframe[variavel2]
    model = sm.OLS(y, X).fit()

    # # Extract the Regression Coefficients
    beta1 = model.params[variavel1]  # Coefficient of the Independent Variable
    beta0 = model.params['const']    # intercept
    std_err = model.bse[variavel1]   # Standard Error of Beta1

    # Display the Regression Results
    print(f'β₁ (Slope): {beta1}')
    print(f'β₀ (Intercept): {beta0}')
    print(f'Erro padrão de β₁: {std_err}')

    # Realizar o z-test entre as duas variáveis
    z_value, p_value_z = ztest(dataframe[variavel1], dataframe[variavel2], value=0)

    # Display the Z-value and the p-value of the test
    print(f'Valor z: {z_value}')
    print(f'p-value do teste z: {p_value_z}')

   # Decision on the Null Hypothesis
    alpha = 0.05  # Significance Level
    if p_value_z < alpha:
        print("Reject the null hypothesis: There is a significant relationship between the variables.")
    else:
        print("Do not reject the null hypothesis: There is no significant relationship between the variables.")


perform_regression_and_ztest(dist_versus_infmutua, 'dist', 'mutual_inf')

β₁ (Slope): -4.795368548459467e-06
β₀ (Intercept): 0.30774109873811734
Erro padrão de β₁: 8.502854439579433e-07
Valor z: 57.04627018390193
p-value do teste z: 0.0
Reject the null hypothesis: There is a significant relationship between the variables.


In [22]:
perform_regression_and_ztest(mob_versus_infmutua, 'mob', 'mutual_inf')

β₁ (Slope): 1.0536852356620077e-05
β₀ (Intercept): 0.27437669951687976
Erro padrão de β₁: 2.7924560484172067e-06
Valor z: 16.198173718186904
p-value do teste z: 5.19470129641266e-59
Reject the null hypothesis: There is a significant relationship between the variables.


In [23]:
# Calcular a correlação de Spearman
corr, p_value = spearmanr(dist_versus_infmutua['dist'], dist_versus_infmutua['mutual_inf'])

print(f"Spearman Correlation Coefficient: {corr}")
print(f"P-value: {p_value}")

Spearman Correlation Coefficient: -0.09487030603206727
P-value: 0.0017923926861957407


In [24]:
# Calcular a correlação de Spearman
corr, p_value = spearmanr(mob_versus_infmutua['mob'], mob_versus_infmutua['mutual_inf'])

print(f"Spearman Correlation Coefficient: {corr}")
print(f"P-value: {p_value}")

Spearman Correlation Coefficient: 0.2681915013512907
P-value: 6.323496978848537e-15


In [25]:
# Calcular a correlação de Kendall
corr, p_value = kendalltau(dist_versus_infmutua['dist'], dist_versus_infmutua['mutual_inf'])

print(f"Kendall Correlation Coefficient: {corr}")
print(f"P-value: {p_value}")

Kendall Correlation Coefficient: -0.06331639731305001
P-value: 0.0018271529456932933


In [26]:
corr, p_value = kendalltau(mob_versus_infmutua['mob'], mob_versus_infmutua['mutual_inf'])

print(f"Kendall Correlation Coefficient: {corr}")
print(f"P-value: {p_value}")

Kendall Correlation Coefficient: 0.1777221747304822
P-value: 3.0541519868561245e-14
